## HW11 — матрица ошибок и метрики классификации

Оцените логистическую регрессию при дисбалансе, исследуйте пороги и метрики. Полные данные не пересчитывались.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
data = pd.read_csv("./processed_train.csv")

In [ ]:
data.head()

### Баланс классов

Посчитайте объекты каждого класса в изменённой таблице и сравните их доли.

In [ ]:
### Your code is here

data['HasDetections'].value_counts()



### Holdout

Разделите признаки и таргет на train/test.

In [ ]:
y = data['HasDetections']
X = data.drop('HasDetections', axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X, y,
                                                     test_size=0.25,
                                                     shuffle=True,
                                                     random_state=1)

Обучите Pipeline со StandardScaler и LogisticRegression с L2, C=0.01. Посчитайте accuracy на train и test.

In [ ]:
### Your code is here

from sklearn.preprocessing import StandardScaler

pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty='l2',C=0.01))])

pipe.fit(X_train, y_train)

print(pipe.score(X_train, y_train))
print(pipe.score(X_test, y_test))

In [ ]:
print(pipe.classes_)

print(pipe.predict(X_test.head(1)))

print(pipe.predict_proba(X_test.head(1)))

print(pipe.decision_function(X_test.head(1)))

Высокая accuracy при дисбалансе сама по себе не доказывает улучшение модели.

### Матрица ошибок

Вычислите вручную TN, FP, FN, TP на test и сравните со sklearn.confusion_matrix. Нулевая метка — отсутствие вируса, единичная — наличие.

In [ ]:
pipe.fit(X_train, y_train)
pred_test = pipe.predict(X_test)

### Your code is here
from sklearn.metrics import confusion_matrix, f1_score

tn, fp, fn, tp = confusion_matrix(y_test, pipe.predict(X_test)).ravel()

print(f'True Negative errors: {tn}')
print(f'False Positive errors: {fp}')
print(f'False Negative errors: {fn}')
print(f'True Positive errors: {tp}')





In [ ]:
print(y.value_counts())

In [ ]:
from sklearn.metrics import confusion_matrix

### Your code is here

accuracy = (tp + tn)/(tn + fp + fn + tp)
accuracy
# acc = metrics.accuracy_score(y_test, pred_test)
# acc




### Accuracy

Посчитайте (TP + TN)/(TP + TN + FP + FN) и сравните со sklearn.

In [ ]:
### Your code is here








Сравните accuracy с ранее полученным результатом.

### Дисбаланс

Константный прогноз большинства может давать высокую accuracy, не обнаруживая редкий положительный класс.

### Precision и Recall

Вычислите TP/(TP+FP) и TP/(TP+FN), затем сравните со sklearn.

In [ ]:
from sklearn.metrics import recall_score, precision_score

### Your code is here
precision = tp / (tp + fp)

recall = tp / (tp + fn)

print(f'Precision: {precision}')
print(f'Recall: {recall}')





Сопоставьте эти метрики с accuracy.

### Порог 0.5

Получите метки сравнением predict_proba положительного класса с 0.5 и сравните с predict.

In [ ]:
### Your code is here

# y_pred = (pipe.predict_proba(X_test)[:, 1] > threshold).astype('float')
y_prediction = pipe.predict_proba(X_test)[:, 1] > 0.5
# y_prediction

tn, fp, fn, tp = confusion_matrix(y_test, y_prediction).ravel()

print(f'True Negative errors: {tn}')
print(f'False Positive errors: {fp}')
print(f'False Negative errors: {fn}')
print(f'True Positive errors: {tp}')

# print(y_prediction==pipe.predict(X_test))


### Ограничение precision

Найдите порог с максимальным recall среди вариантов precision > 0.6.

In [ ]:
from sklearn.metrics import precision_recall_curve

### Строим всевозможные комбинации precision, recall, threshold
precision, recall, thresholds = precision_recall_curve(y_test, pipe.predict_proba(X_test)[:, 1])
# precision, recall, thres = precision_recall_curve(y_test, pred_probs[:, 1])

### Находим такую тройку, при которой recall максимален, а precision > 0.6

ind = np.where((precision[:-1] > 0.6) & (recall[:-1] == recall[:-1][precision[:-1] > 0.6].max()))

print(f"Precison:\t {precision[ind][-1]}")
print(f"Recall: \t {recall[ind][-1]}")
print(f"Threshold:\t {thresholds[ind][-1]}")

Используйте найденный порог для вероятностей положительного класса.

### Ограничение recall

Максимизируйте precision при recall > 0.2.

In [ ]:
### Your code is here

precision, recall, thresholds = precision_recall_curve(y_test, pipe.predict_proba(X_test)[:, 1])
# precision, recall, thres = precision_recall_curve(y_test, pred_probs[:, 1])

### Находим такую тройку, при которой recall максимален, а precision > 0.6

ind = np.where((recall[:-1] > 0.2) & (precision[:-1] == precision[:-1][recall[:-1] > 0.2].max()))

print(f"Precison:\t {precision[ind][-1]}")
print(f"Recall: \t {recall[ind][-1]}")
print(f"Threshold:\t {thresholds[ind][-1]}")






### Комбинации метрик

Найдите пороги максимума среднего арифметического (P+R)/2, геометрического sqrt(P*R) и min(P,R). Выведите P/R и порог, постройте линии уровня через level_lines и отметьте найденные точки.

In [ ]:
def level_lines(func, name):
    x = np.linspace(0, 1.1, 100)
    y = np.linspace(0, 1.1, 100)

    X, Y = np.meshgrid(x, y)
    Z = func(X, Y)

    plt.figure(figsize=(5, 5))
    plt.title(name)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.contour(X, Y, Z, 15, cmap='plasma');


In [ ]:
### Покажем на примере среднего арифметического, что должно у вас получиться

metric_func = lambda x, y: (x + y)/2

arithm_idx = (metric_func(precision[:-1], recall[:-1])).argmax()

print(f"Precision:\t {precision[arithm_idx]}")
print(f"Recall: \t {recall[arithm_idx]}")
print(f"Threshold:\t {thresholds[arithm_idx]}")

level_lines(metric_func, "Arithmetic AVG")
plt.plot(recall[arithm_idx], precision[arithm_idx], 'ro');

In [ ]:
### Повторите для среднего геометрического
### Your code is here
metric_func = lambda x, y: (x*y)**0.5

arithm_idx = (metric_func(precision[:-1], recall[:-1])).argmax()

print(f"Precision:\t {precision[arithm_idx]}")
print(f"Recall: \t {recall[arithm_idx]}")
print(f"Threshold:\t {thresholds[arithm_idx]}")

level_lines(metric_func, "Geometric AVG")
plt.plot(recall[arithm_idx], precision[arithm_idx], 'ro');







In [ ]:
### Повторите для минимума
### Your code is here


metric_func = lambda x, y: np.minimum(x, y)

arithm_idx = (metric_func(precision[:-1], recall[:-1])).argmax()

print(f"Precision:\t {precision[arithm_idx]}")
print(f"Recall: \t {recall[arithm_idx]}")
print(f"Threshold:\t {thresholds[arithm_idx]}")

level_lines(metric_func, "Minimum")
plt.plot(recall[arithm_idx], precision[arithm_idx], 'ro');









### F-мера

Используйте F=(1+beta²)*P*R/(beta²*P+R). Найдите оптимальные пороги для beta²=1, 0.05 и 4; выведите P, R, порог и F, постройте линии уровня. Незавершённые части вывода и графиков не дописаны.

In [ ]:
### Повторите для F-меры при \beta^2=1
### Выведите ее вместе с precision/recall/threshold
### Your code is here


### Повторите для минимума
### Your code is here

betta = 1.0


metric_func = lambda x, y: np.divide(
    (1 + betta**2) * (np.multiply(x, y)),
    (np.multiply(betta**2, x) + y),
    out=np.zeros_like(x + y, dtype=float),
    where=(np.multiply(betta**2, x) + y) != 0
)

arithm_idx = (metric_func(precision[:-1], recall[:-1])).argmax()

print(f"Precision:\t {precision[arithm_idx]}")
print(f"Recall: \t {recall[arithm_idx]}")
print(f"Threshold:\t {thresholds[arithm_idx]}")
level_lines(metric_func, "F1")
plt.plot(recall[arithm_idx], precision[arithm_idx], 'ro');













In [ ]:
import math
import numpy as np

# 1. Корректная лямбда-функция трех переменных по условию
metric_func = lambda p, r, beta: np.divide(
    (1 + beta**2) * (p * r),
    (beta**2 * p + r),
    out=np.zeros_like(p, dtype=float),
    where=(beta**2 * p + r) != 0
)

# 2. Поиск индекса максимума (с ограничением по длины thresholds)
valid_len = len(thresholds)
f_scores = metric_func(precision[:valid_len], recall[:valid_len], 1.0)
arithm_idx = f_scores.argmax()

# Исходные «сырые» значения из ваших массивов
t_raw = thresholds[arithm_idx]
p_raw = precision[arithm_idx]
r_raw = recall[arithm_idx]
f_raw = f_scores[arithm_idx]

# --- ВАРИАНТ А: Округление строго ВВЕРХ (math.ceil) ---
def ceil_3(val):
    return math.ceil(val * 1000) / 1000

print("\nВАРИАНТ Б (Стандартное округление, round):")
print(f"{round(t_raw, 3):.3f}, {round(p_raw, 3):.3f}, {round(r_raw, 3):.3f}, {round(f_raw, 3):.3f}")


In [ ]:
### Повторите для F-меры при \beta^2=0.05
### Выведите ее вместе с precision/recall/threshold
### Your code is here



import math
import numpy as np

# 1. Корректная лямбда-функция трех переменных по условию
metric_func = lambda p, r, beta: np.divide(
    (1 + 0.05) * (p * r),
    (0.05 * p + r),
    out=np.zeros_like(p, dtype=float),
    where=(0.05 * p + r) != 0
)

# 2. Поиск индекса максимума (с ограничением по длины thresholds)
valid_len = len(thresholds)
f_scores = metric_func(precision[:valid_len], recall[:valid_len], 1.0)
arithm_idx = f_scores.argmax()

# Исходные «сырые» значения из ваших массивов
t_raw = thresholds[arithm_idx]
p_raw = precision[arithm_idx]
r_raw = recall[arithm_idx]
f_raw = f_scores[arithm_idx]

# --- ВАРИАНТ А: Округление строго ВВЕРХ (math.ceil) ---
def ceil_3(val):
    return math.ceil(val * 1000) / 1000

print("\nВАРИАНТ Б (Стандартное округление, round):")
print(f"{round(t_raw, 3):.3f}, {round(p_raw, 3):.3f}, {round(r_raw, 3):.3f}, {round(f_raw, 3):.3f}")






In [ ]:
### Повторите для F-меры при \beta^2=4
### Выведите ее вместе с precision/recall/threshold
### Your code is here


import math
import numpy as np

# 1. Корректная лямбда-функция трех переменных по условию
metric_func = lambda p, r, beta: np.divide(
    (1 + 4) * (p * r),
    (4 * p + r),
    out=np.zeros_like(p, dtype=float),
    where=(4 * p + r) != 0
)

# 2. Поиск индекса максимума (с ограничением по длины thresholds)
valid_len = len(thresholds)
f_scores = metric_func(precision[:valid_len], recall[:valid_len], 1.0)
arithm_idx = f_scores.argmax()

# Исходные «сырые» значения из ваших массивов
t_raw = thresholds[arithm_idx]
p_raw = precision[arithm_idx]
r_raw = recall[arithm_idx]
f_raw = f_scores[arithm_idx]

# --- ВАРИАНТ А: Округление строго ВВЕРХ (math.ceil) ---
def ceil_3(val):
    return math.ceil(val * 1000) / 1000

print("\nВАРИАНТ Б (Стандартное округление, round):")
print(f"{round(t_raw, 3):.3f}, {round(p_raw, 3):.3f}, {round(r_raw, 3):.3f}, {round(f_raw, 3):.3f}")










